# MTM extent law refit: θ₉₅(kᵧ, ŝ, q) (Fusion_PhD-bhx7.13)

**Question.** The law `θ₉₅,env = 9.46 kᵧ^−0.351 ŝ^−0.527 [π]` (`results/mtm_kxscan_widefit`: centre 7.129 × margin 1.327, fitted on 24 cells of the 5×5 kx scan) has no q term, and `mtm_extent_parameters` found q matters on the NSTX n1000 hypercube. This notebook refits the law with q on n1000, treats the box-contact modes as censored, sets the margin on n1000, validates on the independent n300 draw, and tests the result on the SPR-008 MTM hypercubes. **No new runs; advisory.**

**Extent.** pyrokinetics `Extent` (95 % rule, full span, in π) of `|φ|² + |A∥|²`, as in `mtm_extent_parameters_mtm_only` (GA `analysis/mtm-only-rescore` 345f6c9), but taken from the cube's `eigenfunctions` (the SPR-008 cubes carry no separate `phi`/`apar`); a check cell compares the two on n1000.

**Parameters** are the cube coordinates (the source the law was fitted on, Fusion_PhD-jmw0: kᵧ in ρ_ref,pyro units; guard: iteration_33 of n1000 reads kᵧ = 0.58085, ŝ = 1.32921). A parameter that is not a coordinate (fixed in that database) is read from the cube's base input.

**Population.** `mode_classification` label `MTM` (T > 0.15, ω < 0; GA master), γ > 0, converged (`growth_rate_tolerance` < 0.01), and not in contact with the θ box (`frac_edge` < 1e-2). Box-contact modes are kept as *censored* (measured extent is a lower bound). Robustness: tearing parity E > 0.5 and ω < 0 instead of the label.

In [ ]:
import os, sys, warnings, json
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from scipy import stats, optimize
from matplotlib.ticker import StrMethodFormatter
from dotenv import load_dotenv
from pyrokinetics import Pyro
from pyrokinetics.diagnostics.extent import Extent
from pyrokinetics.pyroscan import PyroScanGKOutput

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from general_analysis import mode_classification as mc

warnings.filterwarnings("ignore")
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12})  # PLOTTING_AND_PUBLISHING 4a
def plain_ticks(ax):
    """Log axes: 1-2-5 ticks inside the limits, plain {x:g} labels, no minor ticks."""
    for axis, lim, setter in ((ax.xaxis, ax.get_xlim(), ax.set_xticks), (ax.yaxis, ax.get_ylim(), ax.set_yticks)):
        if axis.get_scale() != "log": continue
        t = [m * 10.0 ** e for e in range(int(np.floor(np.log10(lim[0]))) - 1, int(np.ceil(np.log10(lim[1]))) + 1) for m in (1, 2, 5)]
        setter([v for v in t if lim[0] <= v <= lim[1]]); axis.set_major_formatter(StrMethodFormatter("{x:g}"))
        axis.set_minor_formatter(plt.NullFormatter()); axis.set_minor_locator(plt.NullLocator())
        (ax.set_xlim if axis is ax.xaxis else ax.set_ylim)(lim)
load_dotenv(ROOT / "local.env", override=True)
lhc = Path(os.environ["GK_DATA_ROOT"]).expanduser() / "GS2/Runs/LATIN_HYPERCUBE"
plot_dir = ROOT / "Plots" / "mtm_extent_law_refit"
plot_dir.mkdir(parents=True, exist_ok=True)
sets = {"n1000": "NSTX_MTM/beta_q_shat_ky_n1000", "n300": "NSTX_MTM/beta_q_shat_ky_n300",
        # SPR-008 (CSD3 databases, docs/gyro_data_runs.md). MTM_3d_Model1 and MTM_2d_Model3 skipped: median tolerance > 1.
        "4d_Model1_1": "SPR-008/aLTe_aLn_shat_q_n500", "3d_Model2_1": "SPR-008/aLTe_aLn_q_n500",
        "2d_Model1": "SPR-008/aLTe_shat", "2d_Model2": "SPR-008/aLTe_q"}
old = dict(env=9.46, a=-0.351, b=-0.527, margin=1.327)   # results/mtm_kxscan_widefit
TOL, EDGE = 1e-2, 1e-2
coord = dict(ky="ky", shat="shat", q="q", beta="beta", aLTe="electron_temp_gradient", aLne="electron_dens_gradient")

## Load

One cube per database (`pyro_cube_avg`, tail-averaged GS2). T and E from `mc.tearing_parity` (FieldLine on each sample's own Pyro, from the cube's A∥), ω from the cube, extent and edge weight from the eigenfunctions.

In [ ]:
plain = lambda a: a.pint.dequantify() if hasattr(a, "pint") else a

def extent_edge(ds, fields):
    ef = plain(ds["eigenfunctions"]).sel(field=fields)
    power = (abs(ef) ** 2).sum("field").isel(kx=0, ky=0, missing_dims="ignore").squeeze()
    w = PyroScanGKOutput(xr.Dataset({"eigenfunctions_squared": power}))
    Extent(w)
    th, pw = np.asarray(power.theta), np.asarray(power)
    edge = np.trapezoid(pw * (abs(th) > 0.9 * abs(th).max()), th, axis=-1) / np.trapezoid(pw, th, axis=-1)
    return np.asarray(plain(w["extent"])).ravel() / np.pi, edge

def load(db, leaf):
    scan, ds = mc.load_gs2_cube(lhc / leaf, "pyro_cube_avg")
    ds = mc.as_samples(scan, ds)
    base = Pyro(gk_file=lhc / leaf / "pyro_cube_avg/pyroscan_base.input", gk_code="GS2")
    fixed = dict(ky=float(base.numerics.ky.to(base.norms.pyrokinetics.rhoref ** -1).m), shat=float(base.local_geometry.shat), q=float(base.local_geometry.q),
                 beta=float(base.numerics.beta.m), aLTe=float(base.local_species.electron.inverse_lt.m), aLne=float(base.local_species.electron.inverse_ln.m))
    out = pd.DataFrame({"db": db, "run": ds.sample_name.values.astype(str)})
    for k, c in coord.items():
        out[k] = np.broadcast_to(np.asarray(ds[c]).ravel(), len(out)) if c in ds.coords else fixed[k]
    tp = mc.tearing_parity(scan, ds)
    out["T"], out["E"] = np.asarray(tp["T"]).reshape(len(out), -1)[:, 0], np.asarray(tp["E"]).reshape(len(out), -1)[:, 0]
    out["omega"] = mc.ION_DIRECTION * np.asarray(plain(ds["mode_frequency"])).ravel()
    out["gamma"] = np.asarray(plain(ds["growth_rate"])).ravel()
    out["tol"] = np.asarray(plain(ds["growth_rate_tolerance"])).ravel()
    out["extent"], out["edge"] = extent_edge(ds, ["phi", "apar"])
    if "apar" in ds:   # check: eigenfunctions vs the raw fields used by mtm_extent_parameters
        p = (abs(plain(ds["phi"])) ** 2 + abs(plain(ds["apar"])) ** 2).isel(kx=0, ky=0, missing_dims="ignore").squeeze()
        w = PyroScanGKOutput(xr.Dataset({"eigenfunctions_squared": p})); Extent(w)
        d = abs(np.asarray(plain(w["extent"])).ravel() / np.pi / out.extent - 1)
        print(f"{db}: extent from eigenfunctions vs raw fields, max rel diff {np.nanmax(d):.2e}, median {np.nanmedian(d):.2e}")
    print(db, "fixed from base input:", {k: round(v, 5) for k, v in fixed.items() if coord[k] not in ds.coords})
    return out

df = pd.concat([load(db, leaf) for db, leaf in sets.items()], ignore_index=True)
df["label"] = mc.classify_T_omega(df)   # MTM = omega < 0 & T > 0.15, identical to classify(); chi only enters the KBM label
df["tear"] = (df.E > 0.5) & (df.omega < 0)
g33 = df[(df.db == "n1000") & (df.run == "iteration_33")].iloc[0]
assert round(g33.ky, 5) == 0.58085 and round(g33.shat, 5) == 1.32921, (g33.ky, g33.shat)   # Fusion_PhD-jmw0 guard
print("loaded", df.groupby("db").size().to_dict())

## kᵧ on SPR-008

The SPR-008 cubes carry kᵧ as a size-1 coordinate. It should equal the base deck's kᵧ in ρ_ref,pyro units (the law's unit).

In [ ]:
for db, leaf in sets.items():
    if db.startswith("n"): continue
    b = Pyro(gk_file=lhc / leaf / "pyro_cube_avg/pyroscan_base.input", gk_code="GS2")
    print(db, "cube ky", sorted(df[df.db == db].ky.unique()), "| base deck ky [rho_pyro]", float(b.numerics.ky.to(b.norms.pyrokinetics.rhoref ** -1).m))

## Population and counts

Drops in order: not labelled MTM, γ ≤ 0, not converged (tolerance ≥ 0.01 or NaN). The survivors split into **kept** (frac_edge < 1e-2) and **censored** (box contact; extent is a lower bound).

In [ ]:
def gate(d, mtm):
    m = mtm & (d.gamma > 0) & (d.tol < TOL)
    return d[m & (d.edge < EDGE)].assign(cens=False), d[m & (d.edge >= EDGE)].assign(cens=True)

rows = []
for db, d in df.groupby("db", sort=False):
    mtm = d.label == "MTM"
    rows.append(dict(db=db, loaded=len(d), MTM=int(mtm.sum()), MTM_unstable=int((mtm & (d.gamma > 0)).sum()),
                     MTM_unstable_converged=int((mtm & (d.gamma > 0) & (d.tol < TOL)).sum()),
                     kept=len(gate(d, mtm)[0]), censored=len(gate(d, mtm)[1]),
                     no_tol_gate_kept=int((mtm & (d.gamma > 0) & (d.edge < EDGE)).sum()), tear_gate_kept=len(gate(d, d.tear)[0])))
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
pop = {db: pd.concat(gate(d, d.label == "MTM")) for db, d in df.groupby("db", sort=False)}

## Fits

`ln θ₉₅ = c + a ln kᵧ + b ln ŝ + d ln q`. Three treatments of the censored (box-contact) modes: **excluded** (OLS on kept, as `mtm_extent_parameters`), **at measured** (OLS with them at their lower-bound extent), **Tobit** (right-censored Gaussian likelihood: a censored mode contributes P(ln θ > measured)). 95 % CI: t-interval for OLS, 400-sample bootstrap percentile for Tobit.

In [ ]:
def X_of(d, names): return np.column_stack([np.ones(len(d))] + [np.log(d[n].values) for n in names])

def ols(d, names):
    X, y = X_of(d, names), np.log(d.extent.values); n, k = X.shape
    b = np.linalg.lstsq(X, y, rcond=None)[0]; r = y - X @ b
    se = np.sqrt(np.diag(r @ r / (n - k) * np.linalg.inv(X.T @ X)))
    return dict(b=b, ci=stats.t.ppf(0.975, n - k) * se, sigma=np.sqrt(r @ r / (n - k)), n=n, r2=1 - r @ r / ((y - y.mean()) ** 2).sum())

def tobit_fit(X, y, c, start):
    def nll(p):
        z = (y - X @ p[:-1]) / np.exp(p[-1])
        return -(stats.norm.logpdf(z[~c]) - p[-1]).sum() - stats.norm.logsf(z[c]).sum()
    return optimize.minimize(nll, start, method="BFGS").x

def tobit(d, names, nboot=400, seed=0):
    X, y, c = X_of(d, names), np.log(d.extent.values), d.cens.values
    o = ols(d[~d.cens], names); p = tobit_fit(X, y, c, np.r_[o["b"], np.log(o["sigma"])])
    rng = np.random.default_rng(seed)
    boot = np.array([tobit_fit(X[i], y[i], c[i], p) for i in (rng.integers(0, len(y), len(y)) for _ in range(nboot))])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return dict(b=p[:-1], ci=((hi - lo) / 2)[:-1], lo=lo[:-1], hi=hi[:-1], sigma=float(np.exp(p[-1])), n=len(y), ncens=int(c.sum()))

NAMES = ["ky", "shat", "q"]
fmt = lambda f: "  ".join(f"{n} {f['b'][i + 1]:+.3f}±{f['ci'][i + 1]:.3f}" for i, n in enumerate(NAMES)) + f"  | prefactor {np.exp(f['b'][0]):.3f}, sigma {f['sigma']:.3f}, n {f['n']}"
d1 = pop["n1000"]
fits = {"excluded (OLS)": ols(d1[~d1.cens], NAMES), "at measured (OLS)": ols(d1, NAMES), "Tobit (censored)": tobit(d1, NAMES)}
n1t = df[df.db == "n1000"]; tear = pd.concat(gate(n1t, n1t.tear))
nog = n1t[(n1t.label == "MTM") & (n1t.gamma > 0)].assign(cens=lambda d: d.edge >= EDGE)
fits["parity gate, Tobit"] = tobit(tear, NAMES)
fits["no tolerance gate, Tobit"] = tobit(nog, NAMES)
fits["ky-shat only, Tobit"] = tobit(d1, ["ky", "shat"])
for k, f in fits.items(): print(f"{k:28s}", fmt(f) if len(f["b"]) == 4 else f"ky {f['b'][1]:+.3f}±{f['ci'][1]:.3f}  shat {f['b'][2]:+.3f}±{f['ci'][2]:.3f}  sigma {f['sigma']:.3f}")
new = fits["Tobit (censored)"]

## Margin

The envelope is the Tobit centre × M. M is the p-quantile of the residual ln(θ/centre) on n1000, from the Kaplan–Meier estimate with the box-contact modes as right-censored, so a censored mode is never counted as below the envelope it might exceed. The Gaussian value exp(z_p σ) is shown beside it.

In [ ]:
def km_quantile(r, cens, p):
    """p-quantile of right-censored residuals (Kaplan-Meier); NaN if the estimate never reaches p."""
    o = np.argsort(r); r, cens = r[o], cens[o]
    S = np.cumprod(1 - (~cens) / (len(r) - np.arange(len(r))))
    hit = np.nonzero(1 - S >= p)[0]
    return r[hit[0]] if len(hit) else np.nan

centre = lambda d, f, names=NAMES: np.exp(X_of(d, names) @ f["b"])
r1 = np.log(d1.extent.values / centre(d1, new))
margins = {p: dict(km=float(np.exp(km_quantile(r1, d1.cens.values, p))), gauss=float(np.exp(stats.norm.ppf(p) * new["sigma"]))) for p in (0.90, 0.95, 0.99)}
print(pd.DataFrame(margins).round(3).to_string())
M = margins[0.95]["km"]
law = dict(prefactor=float(np.exp(new["b"][0]) * M), centre=float(np.exp(new["b"][0])), margin=M, a=new["b"][1], b=new["b"][2], d=new["b"][3])
print(f"NEW LAW: theta95_env = {law['prefactor']:.2f} ky^{law['a']:.3f} shat^{law['b']:.3f} q^{law['d']:.3f} [pi]  (centre {law['centre']:.3f} x margin {M:.3f})")
env_new = lambda d: law["prefactor"] * d.ky ** law["a"] * d.shat ** law["b"] * d.q ** law["d"]
env_old = lambda d: old["env"] * d.ky ** old["a"] * d.shat ** old["b"]

## Coverage, in sample (n1000) and out of sample (n300)

**below** = kept modes at or below the envelope, as a fraction of kept + censored; **above (sure)** = kept above, plus censored whose lower bound already exceeds it; **undetermined** = censored with lower bound below the envelope. Median measured/envelope is over kept modes.

In [ ]:
def coverage(d, env):
    r = d.extent / env(d); k = ~d.cens
    return dict(n=len(d), ncens=int(d.cens.sum()), below=float(((r <= 1) & k).mean()), above_sure=float((r > 1).mean()),
                undetermined=float(((r <= 1) & d.cens).mean()), median_ratio=float(r[k].median()))
cov = pd.DataFrame({(db, name): coverage(pop[db], env) for db in ("n1000", "n300") for name, env in (("old 9.46", env_old), ("new", env_new))}).T
print(cov.round(3).to_string())
for p, m in margins.items():
    e = lambda d, m=m: env_new(d) / M * m["km"]
    print(f"margin at p={p}: n1000 below {coverage(pop['n1000'], e)['below']:.3f}, n300 below {coverage(pop['n300'], e)['below']:.3f}, n300 above(sure) {coverage(pop['n300'], e)['above_sure']:.3f}")
f300 = tobit(pop["n300"], NAMES)
print("n300 own Tobit fit:", fmt(f300))

## Figure 1. Measured extent against the law: n1000 (in sample) and n300 (out of sample), old and new law

y = x is the envelope. Circles: kept; triangles: box contact (lower bound).

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 10), sharex="col", sharey=True)
for i, db in enumerate(("n1000", "n300")):
    d = pop[db]
    for j, (name, env) in enumerate((("old law", env_old), ("new law", env_new))):
        ax = axs[i, j]; x = env(d)
        ax.scatter(x[~d.cens], d.extent[~d.cens], s=10, alpha=0.5, c="C0")
        ax.scatter(x[d.cens], d.extent[d.cens], s=22, marker="^", c="C3", alpha=0.8)
        ax.set_xscale("log"); ax.set_yscale("log")
        lim = np.array([2, 80]); ax.set_xlim(lim); ax.set_ylim(lim); ax.plot(lim, lim, "k", lw=1)
        ax.set_title(f"{db}, {name}"); plain_ticks(ax)
        if i == 1: ax.set_xlabel(r"envelope $\theta_{95}$ [$\pi$]")
        if j == 0: ax.set_ylabel(r"measured $\theta_{95}$ [$\pi$]")
fig.tight_layout(); fig.savefig(plot_dir / "fig1_measured_vs_law.png")

## Figure 2. q dependence on n1000: partial residual

ln θ₉₅ minus the kᵧ and ŝ terms of the new (Tobit) fit, against q. The line is the fitted q term.

In [ ]:
d = d1; rq = np.log(d.extent) - new["b"][0] - new["b"][1] * np.log(d.ky) - new["b"][2] * np.log(d.shat)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(d.q[~d.cens], rq[~d.cens], s=10, alpha=0.5, c="C0"); ax.scatter(d.q[d.cens], rq[d.cens], s=22, marker="^", c="C3", alpha=0.8)
qs = np.linspace(d.q.min(), d.q.max(), 50); ax.plot(qs, new["b"][3] * np.log(qs), "k", lw=1.5)
ax.set_xscale("log"); ax.set_xlabel("$q$"); ax.set_ylabel(r"$\ln\theta_{95}$ $-$ $k_y$, $\hat s$ terms"); ax.set_title("n1000"); plain_ticks(ax)
fig.tight_layout(); fig.savefig(plot_dir / "fig2_q_partial_residual.png")

## SPR-008

Same gates on the four SPR-008 databases, pooled (one equilibrium, kᵧ fixed). (a) Fit ŝ and q exponents on SPR-008 alone; (b) partial slopes of a/L_Te and a/L_ne after the new law's ŝ and q terms, and the full regression; (c) the new law with its NSTX prefactor on SPR-008: coverage and median ratio, i.e. the SPR-008 prefactor relative to NSTX's.

In [ ]:
spr = pd.concat([pop[k] for k in sets if not k.startswith("n")], ignore_index=True)
print(counts[~counts.db.str.startswith("n")].to_string(index=False))
print("SPR-008 pooled: n", len(spr), "censored", int(spr.cens.sum()), "| ky values", spr.ky.unique())
f_sq = tobit(spr, ["shat", "q"])
print(f"(a) SPR-008 Tobit: shat {f_sq['b'][1]:+.3f}±{f_sq['ci'][1]:.3f} (NSTX {law['b']:+.3f}), q {f_sq['b'][2]:+.3f}±{f_sq['ci'][2]:.3f} (NSTX {law['d']:+.3f}), sigma {f_sq['sigma']:.3f}")
for k in [k for k in sets if not k.startswith("n")]:
    dk = pop[k]; v = [n for n in ("shat", "q") if dk[n].nunique() > 1]
    if len(dk) > 10 and v:
        f = tobit(dk, v, nboot=200); print(f"    {k} (n={len(dk)}, cens {int(dk.cens.sum())}):", "  ".join(f"{n} {f['b'][i + 1]:+.3f}±{f['ci'][i + 1]:.3f}" for i, n in enumerate(v)))
k_ = ~spr.cens
res = np.log(spr.extent) - law["b"] * np.log(spr.shat) - law["d"] * np.log(spr.q)
for n in ("aLTe", "aLne"):
    s = stats.linregress(np.log(spr[n][k_]), res[k_]); print(f"(b) partial slope vs ln {n} (kept only, n={k_.sum()}): {s.slope:+.3f}±{1.96 * s.stderr:.3f}")
f_all = tobit(spr, ["shat", "q", "aLTe", "aLne"])
print("(b) SPR-008 Tobit, all four:", "  ".join(f"{n} {f_all['b'][i + 1]:+.3f}±{f_all['ci'][i + 1]:.3f}" for i, n in enumerate(["shat", "q", "aLTe", "aLne"])))
cs = coverage(spr, env_new); co = coverage(spr, env_old)
print(f"(c) new law on SPR-008: {cs}\n    old law on SPR-008: {co}")
print(f"    SPR-008 centre / NSTX centre (median measured / new centre, kept): {np.median(spr.extent[k_] / (env_new(spr)[k_] / M)):.3f}")

## Figure 3. SPR-008: measured extent against the new law, coloured by gradient

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)
x = env_new(spr)
for ax, n, lab in zip(axs, ("aLTe", "aLne"), ("$a/L_{Te}$", "$a/L_{ne}$")):
    sc = ax.scatter(x[k_], spr.extent[k_], s=12, c=spr[n][k_], cmap="viridis", alpha=0.8)
    ax.scatter(x[~k_], spr.extent[~k_], s=26, marker="^", c=spr[n][~k_], cmap="viridis", norm=sc.norm, edgecolors="r", linewidths=0.6)
    ax.set_xscale("log"); ax.set_yscale("log")
    lim = np.array([min(x.min(), spr.extent.min()) * 0.9, max(x.max(), spr.extent.max()) * 1.1]); ax.set_xlim(lim); ax.set_ylim(lim); ax.plot(lim, lim, "k", lw=1)
    ax.set_xlabel(r"new envelope $\theta_{95}$ [$\pi$]"); ax.set_title("SPR-008"); plain_ticks(ax); fig.colorbar(sc, ax=ax, label=lab)
axs[0].set_ylabel(r"measured $\theta_{95}$ [$\pi$]")
fig.savefig(plot_dir / "fig3_spr008_measured_vs_law.png")   # paper.mplstyle: constrained layout, which colorbars need

In [ ]:
summary = dict(counts=counts.to_dict("records"), law=law, margins=margins,
               fits={k: dict(b=list(map(float, f["b"])), ci=list(map(float, f["ci"])), sigma=f["sigma"], n=f["n"]) for k, f in fits.items()},
               n300_fit=dict(b=list(map(float, f300["b"])), ci=list(map(float, f300["ci"])), n=f300["n"]),
               coverage={f"{a}|{b}": v for (a, b), v in cov.to_dict("index").items()},
               spr008=dict(n=len(spr), ncens=int(spr.cens.sum()), shat_q_fit=dict(b=list(map(float, f_sq["b"])), ci=list(map(float, f_sq["ci"]))),
                           all4_fit=dict(b=list(map(float, f_all["b"])), ci=list(map(float, f_all["ci"]))), cov_new=cs, cov_old=co))
json.dump(summary, open(plot_dir / "summary.json", "w"), indent=1, default=float)